In [1]:
!pip install protobuf==3.19.6
!pip install tensorflow==2.17.0
!pip install mediapipe==0.10.20

  Using cached protobuf-3.19.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (787 bytes)
Using cached protobuf-3.19.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.1 MB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.7
    Uninstalling protobuf-4.25.7:
      Successfully uninstalled protobuf-4.25.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.10 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 3.19.6 which is incompatible.
google-cloud-aiplatform 1.74.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 3.19.6 which is incompatible.
google-cloud-bigquery-connection 1.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from scipy.signal import savgol_filter
from IPython.display import clear_output

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
from IPython.display import clear_output

# MediaPipe setup
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(model_complexity=2,
                    min_detection_confidence=0.7,
                    min_tracking_confidence=0.7)
# FSM và bộ đếm
angle_history = deque(maxlen=30)
fsm_state = "Idle"
counter = 0
hold_frames = 0

# FSM logic cho push-up
def update_fsm(avg_angle, velocity):
    global fsm_state, counter, hold_frames
    if fsm_state == "Idle":
        if avg_angle < 95 and velocity < -1:
            fsm_state = "Down"
    elif fsm_state == "Down":
        if avg_angle > 130 and velocity > 1:
            fsm_state = "Up"
            counter += 1
    elif fsm_state == "Up":
        if avg_angle > 130:
            hold_frames += 1
            if hold_frames >= 5:
                fsm_state = "Idle"
                hold_frames = 0
        else:
            hold_frames = 0

# Hàm tính góc
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - \
              np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

# Mở video
cap = cv2.VideoCapture('/kaggle/input/pseudo-pushup2/IMG_15293.mp4')  # <-- đổi đường dẫn video của bạn
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter('pushup_output.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

with mp_pose.Pose(min_detection_confidence=0.5,
                  min_tracking_confidence=0.5) as pose:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        image = frame.copy()

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Vẽ khung xương
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

            # Lấy các điểm: vai - khuỷu - cổ tay bên phải
            shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x * width,
                        landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y * height]
            elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x * width,
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y * height]
            wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x * width,
                     landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y * height]

            # Tính góc
            raw_angle = calculate_angle(shoulder, elbow, wrist)
            angle_history.append(raw_angle)

            if len(angle_history) >= 5:
                smoothed = savgol_filter(list(angle_history), 5, 2)
                avg_angle = np.mean(smoothed[-15:])
                velocity = smoothed[-1] - smoothed[-2]
            else:
                avg_angle = raw_angle
                velocity = 0

            # Cập nhật trạng thái và đếm
            update_fsm(avg_angle, velocity)

            # Hiển thị kết quả
            cv2.rectangle(image, (0, 0), (300, 110), (0, 0, 0), -1)
            cv2.putText(image, f'Reps: {counter}', (10, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(image, f'State: {fsm_state}', (10, 70),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2)
            cv2.putText(image, f'Angle: {int(avg_angle)}', (10, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

            # # Biểu đồ
            # fig, ax = plt.subplots(figsize=(3, 1.5), dpi=100)
            # ax.plot(list(angle_history), color='orange')
            # ax.set_ylim(0, 180)
            # ax.axis('off')
            # fig.canvas.draw()
            # chart_img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
            # chart_img = chart_img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
            # plt.close(fig)
            # chart_img = cv2.resize(chart_img, (200, 100))
            # image[10:110, width-210:width-10] = chart_img

        #out.write(image)
        clear_output(wait=True)
        plt.imshow(image)
        plt.axis('off')
        plt.pause(0.01)  # Pause to update the plot


    cap.release()
    out.release()
    print("✅ Video đã lưu thành pushup_output.mp4")


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
from IPython.display import clear_output

# MediaPipe setup
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# FSM và bộ đếm
angle_history = deque(maxlen=30)
fsm_state = "Idle"
counter = 0
hold_frames = 0

# FSM logic cho push-up
def update_fsm(avg_angle, velocity):
    global fsm_state, counter, hold_frames
    if fsm_state == "Idle":
        if avg_angle < 95 and velocity < -1:
            fsm_state = "Down"
    elif fsm_state == "Down":
        if avg_angle > 130 and velocity > 1:
            fsm_state = "Up"
            counter += 1
    elif fsm_state == "Up":
        if avg_angle > 130:
            hold_frames += 1
            if hold_frames >= 5:
                fsm_state = "Idle"
                hold_frames = 0
        else:
            hold_frames = 0

# Hàm tính góc
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - \
              np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

# Mở video
cap = cv2.VideoCapture('/kaggle/input/push-up-video/VID_20250408_173931 - Trim.mp4')  # <-- đổi đường dẫn video của bạn
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter('pushup_output.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

with mp_pose.Pose(min_detection_confidence=0.5,
                  min_tracking_confidence=0.5) as pose:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        image = frame.copy()

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Vẽ khung xương
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

            # Lấy các điểm: vai - khuỷu - cổ tay bên phải
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x * width,
                        landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y * height]
            elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x * width,
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y * height]
            wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x * width,
                     landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y * height]

            # Tính góc
            raw_angle = calculate_angle(shoulder, elbow, wrist)
            angle_history.append(raw_angle)

            if len(angle_history) >= 5:
                smoothed = savgol_filter(list(angle_history), 5, 2)
                avg_angle = np.mean(smoothed[-15:])
                velocity = smoothed[-1] - smoothed[-2]
            else:
                avg_angle = raw_angle
                velocity = 0

            # Cập nhật trạng thái và đếm
            update_fsm(avg_angle, velocity)

            # Hiển thị kết quả
            cv2.rectangle(image, (0, 0), (300, 110), (0, 0, 0), -1)
            cv2.putText(image, f'Reps: {counter}', (10, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(image, f'State: {fsm_state}', (10, 70),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2)
            cv2.putText(image, f'Angle: {int(avg_angle)}', (10, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

            # Biểu đồ
            fig, ax = plt.subplots(figsize=(3, 1.5), dpi=100)
            ax.plot(list(angle_history), color='orange')
            ax.set_ylim(0, 180)
            ax.axis('off')
            fig.canvas.draw()
            chart_img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
            chart_img = chart_img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
            plt.close(fig)
            chart_img = cv2.resize(chart_img, (200, 100))
            image[10:110, width-210:width-10] = chart_img

        out.write(image)
        clear_output(wait=True)
        # plt.imshow(image)
        # plt.axis('off')
        # plt.pause(0.01)  # Pause to update the plot


    cap.release()
    out.release()
    print("✅ Video đã lưu thành pushup_output.mp4")


**Thêm check đang tập/nghĩ**

In [8]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
from IPython.display import clear_output

# ========== SETUP ==========
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

angle_history = deque(maxlen=30)
shoulder_history = deque(maxlen=240)  # ~2s nếu fps ~30

fsm_state = "Idle"
counter = 0
hold_frames = 0

# FSM đếm rep
def update_fsm(avg_angle, velocity, is_resting):
    global fsm_state, counter, hold_frames
    if is_resting:
        return  # không cập nhật khi nghỉ

    if fsm_state == "Idle":
        if avg_angle < 100 and velocity < -1:
            fsm_state = "Down"
    elif fsm_state == "Down":
        if avg_angle > 130 and velocity > 1:
            fsm_state = "Up"
            counter += 1
    elif fsm_state == "Up":
        if avg_angle > 130:
            hold_frames += 1
            if hold_frames >= 5:
                fsm_state = "Idle"
                hold_frames = 0
        else:
            hold_frames = 0

# Tính góc
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - \
              np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

# Kiểm tra tư thế nghỉ
def is_resting_pose_by_shoulder(height, threshold_ratio=0.5, percent_frames_above=0.8):
    if len(shoulder_history) < shoulder_history.maxlen:
        return False
    l_y = [pair[0] for pair in shoulder_history]
    r_y = [pair[1] for pair in shoulder_history]
    l_std = np.std(l_y)
    r_std = np.std(r_y)
    low_movement = l_std < 100 and r_std < 100
    threshold_y = height * threshold_ratio
    count_above = sum(1 for ly, ry in zip(l_y, r_y) if ly < threshold_y and ry < threshold_y)
    high_position = (count_above / shoulder_history.maxlen) >= percent_frames_above
    return low_movement or high_position

# Video vào và ra
cap = cv2.VideoCapture('/kaggle/input/push-up-3/Cut_20250427_225427.mov')  # đổi path nếu cần
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
out = cv2.VideoWriter('pushup_output3.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

with mp_pose.Pose(min_detection_confidence=0.5,
                  min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        image = frame.copy()

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # === Làm mờ mặt ===
            try:
                keypoints = [
                    mp_pose.PoseLandmark.NOSE,
                    mp_pose.PoseLandmark.LEFT_EYE,
                    mp_pose.PoseLandmark.RIGHT_EYE,
                    mp_pose.PoseLandmark.LEFT_EAR,
                    mp_pose.PoseLandmark.RIGHT_EAR
                ]
                x_coords = [landmarks[pt.value].x for pt in keypoints]
                y_coords = [landmarks[pt.value].y for pt in keypoints]
                x_min = int(min(x_coords) * width) - 40
                x_max = int(max(x_coords) * width) + 40
                y_min = int(min(y_coords) * height) - 60
                y_max = int(max(y_coords) * height) + 60
                x_min = max(0, x_min)
                y_min = max(0, y_min)
                x_max = min(width, x_max)
                y_max = min(height, y_max)
                face_region = image[y_min:y_max, x_min:x_max]
                if face_region.size > 0:
                    k = max(3, int(min(face_region.shape[:2]) * 0.4) | 1)
                    blurred_face = cv2.GaussianBlur(face_region, (k, k), 0)
                    image[y_min:y_max, x_min:x_max] = blurred_face
            except Exception as e:
                print("❗ Không làm mờ được mặt:", e)

            # === Tính góc tay phải ===
            shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x * width,
                        landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y * height]
            elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x * width,
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y * height]
            wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x * width,
                     landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y * height]

            raw_angle = calculate_angle(shoulder, elbow, wrist)
            angle_history.append(raw_angle)

            if len(angle_history) >= 5:
                smoothed = savgol_filter(list(angle_history), 5, 2)
                avg_angle = np.mean(smoothed[-15:])
                velocity = smoothed[-1] - smoothed[-2]
            else:
                avg_angle = raw_angle
                velocity = 0

            # Lưu vai vào buffer
            left_shoulder_y = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y * height
            right_shoulder_y = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y * height
            shoulder_history.append((left_shoulder_y, right_shoulder_y))

            # Kiểm tra tư thế nghỉ
            is_resting = is_resting_pose_by_shoulder(height)

            # Cập nhật FSM
            update_fsm(avg_angle, velocity, is_resting)

            # Vẽ khung xương
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

            # Vẽ thông tin
            # === Hiển thị thông tin chuyên nghiệp hơn ===
            info_x = width - 360
            info_y = height // 2 - 100
            
            cv2.rectangle(image, (info_x, info_y), (info_x + 340, info_y + 160), (0, 0, 0), -1)
            
            cv2.putText(image, f'Repetition Count: {counter}', (info_x + 10, info_y + 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            
            cv2.putText(image, f'FSM State: {fsm_state}', (info_x + 10, info_y + 70),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 0), 2)
            
            cv2.putText(image, f'Workout Mode: {"Resting" if is_resting else "Working"}',
                        (info_x + 10, info_y + 105), cv2.FONT_HERSHEY_SIMPLEX, 0.75,
                        (255, 100, 255), 2)
            
            cv2.putText(image, f'Elbow Angle: {int(avg_angle)}', (info_x + 10, info_y + 140),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
        out.write(image)
        # clear_output(wait=True)
        # plt.imshow(image)
        # plt.axis('off')
        # plt.pause(0.01)  # Pause to update the plot


    cap.release()
    out.release()
    print("✅ Video đã lưu thành pushup_output.mp4")


✅ Video đã lưu thành pushup_output.mp4
